# Subgraph with Shared Schema

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from dotenv import load_dotenv

load_dotenv()

@tool
def get_weather(location: str):
    """Fetch weather information for a location."""
    if location.lower() == "munich":
        return "It's 15 degrees Celsius and cloudy."
    return "It's 30 degrees and sunny."

tools = [get_weather]
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash").bind_tools(tools)



In [2]:
def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    return "tools" if last_message.tool_calls else END

subgraph_builder = StateGraph(MessagesState)

subgraph_builder.add_node("agent", call_model)
subgraph_builder.add_node("tools", ToolNode(tools=tools))

subgraph_builder.add_conditional_edges("agent", should_continue)
subgraph_builder.add_edge("tools", "agent")

subgraph_builder.set_entry_point("agent")

subgraph = subgraph_builder.compile()

In [4]:
def start_node(state: MessagesState):
    return state

main_graph = StateGraph(MessagesState)

main_graph.add_node("start", start_node)
main_graph.add_node("subgraph", subgraph)

main_graph.add_edge("start", "subgraph")

main_graph.set_entry_point("start")

graph = main_graph.compile()

In [8]:
initial_state = {"messages": [HumanMessage(content="How is the weather in Munich?")]}
result = graph.invoke(initial_state)
print(result)

{'messages': [HumanMessage(content='How is the weather in Munich?', additional_kwargs={}, response_metadata={}, id='6a28ba6d-d929-4562-8008-a16dc6dfd336'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Munich"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--34b4d97b-2ae1-42df-b1a9-92ff949a9325-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Munich'}, 'id': 'ae07003b-9015-4b40-a78f-c414d90ab6f7', 'type': 'tool_call'}], usage_metadata={'input_tokens': 47, 'output_tokens': 16, 'total_tokens': 121, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 58}}), ToolMessage(content="It's 15 degrees Celsius and cloudy.", name='get_weather', id='e39c80f2-319f-4df4-b28b-5dbf0b24d454', tool_call_id='ae07003b-9015-4b40-a78f-c414d90ab6f7'), AIMessage(content="It's 15 deg

In [9]:
# {'messages': [HumanMessage(content='How is the weather in Munich?', additional_kwargs={}, response_metadata={}, id='f346f5a4-a562-47cf-a8c9-81b258cd7733'), 
# AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Munich"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--e5875ec9-5b19-4add-8500-d2cc9c885d7e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Munich'}, 'id': '0e6c3642-3680-495c-a136-fcb7dc698cb7', 'type': 'tool_call'}], usage_metadata={'input_tokens': 47, 'output_tokens': 16, 'total_tokens': 113, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 50}}), 
# ToolMessage(content="It's 15 degrees Celsius and cloudy.", name='get_weather', id='7208e187-0c8d-49eb-914d-871014149a9a', tool_call_id='0e6c3642-3680-495c-a136-fcb7dc698cb7'), 
# AIMessage(content="It's 15 degrees Celsius and cloudy in Munich.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--799fdc4f-15d0-46a5-8ee3-626702fbb19e-0', usage_metadata={'input_tokens': 88, 'output_tokens': 13, 'total_tokens': 195, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 94}})]}

# Subgraph with Different Schema

### Restart the kernel

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from dotenv import load_dotenv

load_dotenv()

@tool
def get_weather(location: str):
    """Fetch weather information for a location."""
    if location.lower() == "munich":
        return "It's 15 degrees Celsius and cloudy."
    return "It's 30 degrees and sunny."

tools = [get_weather]
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash").bind_tools(tools)



In [2]:
def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    return "tools" if last_message.tool_calls else END

In [3]:


subgraph_builder = StateGraph(MessagesState)

subgraph_builder.add_node("agent", call_model)
subgraph_builder.add_node("tools", ToolNode(tools=tools))

subgraph_builder.add_conditional_edges("agent", should_continue)
subgraph_builder.add_edge("tools", "agent")

subgraph_builder.set_entry_point("agent")

subgraph = subgraph_builder.compile()

In [4]:
from typing_extensions import TypedDict

class ParentState(TypedDict):
    parent_messages: list[str]

def start_node(state: ParentState):
    return state

def invoke_subgraph(state: ParentState):
    subgraph_output = subgraph.invoke({"messages": state["parent_messages"]})
    state["parent_messages"] = subgraph_output["messages"]
    return state

main_graph = StateGraph(ParentState)

main_graph.add_node("start", start_node)
main_graph.add_node("invoke_subgraph", invoke_subgraph)

main_graph.add_edge("start", "invoke_subgraph")

main_graph.set_entry_point("start")

graph = main_graph.compile()

In [5]:
initial_state = {
    "parent_messages": [HumanMessage(content="What's the weather in NY?")]
}
result = graph.invoke(initial_state)
print(result)

{'parent_messages': [HumanMessage(content="What's the weather in NY?", additional_kwargs={}, response_metadata={}, id='63a063a4-fd9a-4672-943d-8fecfa006a95'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "NY"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--1c71dad6-227d-4d9c-a033-24cf3e73b5ee-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'NY'}, 'id': 'fb4e0eb1-53e6-41fb-8816-07d01ed1418e', 'type': 'tool_call'}], usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 124, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 61}}), ToolMessage(content="It's 30 degrees and sunny.", name='get_weather', id='90bf4151-ae5a-4552-95dd-5e43ad4fe90e', tool_call_id='fb4e0eb1-53e6-41fb-8816-07d01ed1418e'), AIMessage(content='The weather in NY is 30 d

In [ ]:
{'parent_messages': [HumanMessage(content="What's the weather in NY?", additional_kwargs={}, response_metadata={}, id='63a063a4-fd9a-4672-943d-8fecfa006a95'), 
AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "NY"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--1c71dad6-227d-4d9c-a033-24cf3e73b5ee-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'NY'}, 'id': 'fb4e0eb1-53e6-41fb-8816-07d01ed1418e', 'type': 'tool_call'}], usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 124, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 61}}), 
ToolMessage(content="It's 30 degrees and sunny.", name='get_weather', id='90bf4151-ae5a-4552-95dd-5e43ad4fe90e', tool_call_id='fb4e0eb1-53e6-41fb-8816-07d01ed1418e'), 
AIMessage(content='The weather in NY is 30 degrees and sunny.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--56cc67f4-04f4-4da5-a65d-5373b89ae69d-0', usage_metadata={'input_tokens': 87, 'output_tokens': 12, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}})]}